In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# DOSYA BUL (OTOMATİK)
possible_paths = [
    "Otel_Operasyon_Master.xlsx",
    r"C:\Users\DeLL\Desktop\Otel_Operasyon_Master.xlsx",
    r"C:\Users\DeLL\Downloads\Otel_Operasyon_Master.xlsx"
]

file_path = None
for path in possible_paths:
    if os.path.exists(path):
        file_path = path
        break

if file_path is None:
    raise FileNotFoundError("Excel dosyası bulunamadı!")

print(f"Dosya bulundu: {file_path}")

# VERİYİ YÜKLE

df_res = pd.read_excel(file_path, sheet_name="1_Rezervasyon_Gelir")

# VERİ HAZIRLAMA

# Tarih oluştur
df_res['arrival_date'] = pd.to_datetime(
    df_res['arrival_date_year'].astype(str) + "-" +
    df_res['arrival_date_month'].astype(str) + "-01",
    errors="coerce"
)

# Toplam gece
df_res["total_nights"] = df_res["stays_in_weekend_nights"] + df_res["stays_in_week_nights"]

## KPI'LAR

total_revenue = df_res["adr"].sum()
avg_adr = df_res["adr"].mean()
cancel_rate = df_res["is_canceled"].mean() * 100
repeat_rate = df_res["is_repeated_guest"].mean() * 100
avg_stay = df_res["total_nights"].mean()

print("\n===== KPI =====")
print(f"Toplam Gelir: {total_revenue:,.2f}")
print(f"Ortalama ADR: {avg_adr:.2f}")
print(f"İptal Oranı: {cancel_rate:.2f}%")
print(f"Tekrar Eden Müşteri: {repeat_rate:.2f}%")
print(f"Ortalama Konaklama: {avg_stay:.2f} gün")

## AYLIK GELİR TRENDİ (İPTAL HARİÇ)

monthly_revenue = (
    df_res[df_res["is_canceled"] == 0]
    .groupby(df_res["arrival_date"].dt.to_period("M"))["adr"]
    .sum()
)

plt.figure(figsize=(12,5))
sns.lineplot(
    x=monthly_revenue.index.astype(str),
    y=monthly_revenue.values,
    marker="o"
)
plt.xticks(rotation=45)
plt.title("Aylık Gelir Trendi (İptal Hariç)")
plt.xlabel("Ay")
plt.ylabel("Toplam Gelir")
plt.tight_layout()
plt.show()

## SEGMENT ANALİZ

revenue_by_segment = df_res.groupby("market_segment")["adr"].sum().sort_values(ascending=False)
print("\nSegment Gelir:\n", revenue_by_segment)

## CUSTOMER TYPE ANALİZ

revenue_by_customer_type = df_res.groupby("customer_type")["adr"].sum().sort_values(ascending=False)
print("\nCustomer Type Gelir:\n", revenue_by_customer_type)

## ORTALAMA HARCAMA & KONAKLAMA (SEGMENT BAZLI)

avg_spend_segment = df_res.groupby("market_segment")["adr"].mean()
avg_stay_segment = df_res.groupby("market_segment")["total_nights"].mean()

avg_df = pd.DataFrame({
    "Avg_Spend": avg_spend_segment,
    "Avg_Stay": avg_stay_segment
})

print("\nSegment Bazlı Ortalama Harcama ve Konaklama:\n", avg_df)

## KANAL ANALİZİ

channel_counts = df_res["distribution_channel"].value_counts()
channel_revenue = df_res.groupby("distribution_channel")["adr"].sum().sort_values(ascending=False)

print("\nRezervasyon Kanal Dağılımı:\n", channel_counts)
print("\nKanal Bazlı Gelir:\n", channel_revenue)

## ÜLKE ANALİZİ

top_countries = df_res["country"].value_counts().head(10)
print("\nTop 10 Ülke:\n", top_countries)

## GRAFİKLER

fig, axes = plt.subplots(2, 2, figsize=(14,10))

# Segment
revenue_by_segment.plot(kind="bar", ax=axes[0,0])
axes[0,0].set_title("Segment Gelir")
axes[0,0].tick_params(axis='x', rotation=45)

# Customer type
revenue_by_customer_type.plot(kind="bar", ax=axes[0,1])
axes[0,1].set_title("Customer Type Gelir")
axes[0,1].tick_params(axis='x', rotation=45)

# Kanal
channel_revenue.plot(kind="bar", ax=axes[1,0])
axes[1,0].set_title("Kanal Gelir")
axes[1,0].tick_params(axis='x', rotation=45)

# Ülke
top_countries.plot(kind="bar", ax=axes[1,1])
axes[1,1].set_title("Top 10 Ülke")
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()